# Lab 07  - Chain of Thought: Step by Step, Self Consistency, Private vs Exposed Rationales

**Week 3 - Prompt Engineering and Task to Prompt Mapping**

**Focus.** Apply Chain of Thought (CoT) prompting to math word problems and policy logic, compare against a no-CoT baseline, use self consistency (sample and select) to stabilise answers, and control whether rationales are private (hidden) or exposed.

## Outcomes
By the end you can:
1. Use a CoT trigger in an instruction prompt and see its effect on accuracy.
2. Implement self consistency by sampling several candidates and selecting a consensus answer.
3. Control rationale visibility, private versus exposed, and enforce it with a guard.
4. Measure accuracy across no-CoT, single CoT, and consensus CoT, and reason about the token and latency trade-offs.

## How this lab runs

You will not paste model output by hand. Instead you call a small, offline **teaching provider** that stands in for a hosted LLM. It is deterministic and seeded, so your results are reproducible and the lab runs with no API key and no network.

Two things to hold onto:

- The provider is driven by **your prompt**. A step-by-step trigger turns its reasoning path on. A rationale field in the contract (with no instruction to hide reasoning) makes it return a rationale. If you forget the trigger, you get the shallow no-CoT behaviour, and your accuracy drops. Prompt wording has real, measurable consequences here.
- The `check(...)` helper is **soft**. A missing or wrong implementation shows up as `[FAIL]`, never a crash, so you can run the whole notebook top to bottom at any point and watch your pass count climb.

The provider is a stand-in, not a language model. The numbers it produces are illustrative of well documented behaviour (shallow answers miss multi-step problems; sampling and voting reduces slips), not measurements of any specific model. The harness you build here is exactly what you would run against a real model, and the final cell shows the one function you would swap to do so.

> **Currency flags (2026).** Read before you lean on this in production.
> - Reasoning models (for example the o-series, or Claude with extended thinking) already reason internally, so an explicit "think step by step" trigger matters less for them, and several restrict sampling controls such as temperature. Self consistency still applies wherever you can draw independent samples. Confirm per provider.
> - Sampling `temperature` and any `seed` are best effort on hosted endpoints and are not a reproducibility guarantee. The determinism in this lab comes from the local provider, not from a hosted model.
> - You can enforce the JSON contract far more reliably with structured outputs than with prose instructions. On the Claude API structured outputs are now generally available through `output_config.format` with a JSON schema, no beta header required (the older `structured-outputs-2025-11-13` beta header still works during the transition). OpenAI offers the same idea via strict `json_schema` response formats. Prefer these over "return STRICT JSON" wording when the platform supports them.

In [ ]:
# GIVEN - setup. Do not edit this cell.
import json, os, re, hashlib
from collections import Counter

# The dataset ships as cot_tasks.json. An identical embedded copy is used as a
# cold-start fallback so this notebook runs even if the file is not next to it.
_EMBEDDED = {
    "mwp": [
        {
            "id": "M01",
            "text": "A jar has 18 red marbles and 12 blue marbles. If you remove 7 red and 3 blue, how many marbles remain?"
        },
        {
            "id": "M02",
            "text": "A train travels 120 km in 2 hours, then 150 km in 3 hours. What is its average speed in km per hour for the whole trip?"
        },
        {
            "id": "M03",
            "text": "There are 5 boxes with 8 pens each. You give away 9 pens, then buy 7 more. How many pens do you have?"
        },
        {
            "id": "M04",
            "text": "A baker made 48 cookies. She sells three quarters of them in the morning and 8 more in the afternoon. How many cookies are left?"
        },
        {
            "id": "M05",
            "text": "Two numbers sum to 30. One is 8 more than the other. What is the larger number?"
        }
    ],
    "pol": [
        {
            "id": "P01",
            "text": "Policy: Reimburse taxi rides only if the receipt timestamp is between 06:00 and 22:00 local and the total is at most 50 dollars. Claim: 47 dollars at 21:40 with a valid receipt. Should it be reimbursed?"
        },
        {
            "id": "P02",
            "text": "Policy: The travel meal stipend is 20 dollars per day and alcohol is never reimbursable. Claim: one receipt showing lunch 14 dollars and beer 6 dollars. What is reimbursable?"
        },
        {
            "id": "P03",
            "text": "Policy: Laptops must be approved by IT before purchase. Claim: an employee bought a 900 dollar laptop without prior approval. Is it eligible?"
        },
        {
            "id": "P04",
            "text": "Policy: Conference fees are reimbursable only if the manager pre-approves by email before the purchase. Claim: the manager approved by email the day after the purchase. Is it eligible?"
        },
        {
            "id": "P05",
            "text": "Policy: Rideshare tips are reimbursable only when the combined total of fare plus tip is at most 40 dollars. Claim: fare 32 dollars and tip 9 dollars. Is it eligible?"
        }
    ],
    "gold": {
        "M01": 20,
        "M02": 54,
        "M03": 38,
        "M04": 4,
        "M05": 19,
        "P01": "approve",
        "P02": "reimburse_meal_only",
        "P03": "deny",
        "P04": "deny",
        "P05": "deny"
    }
}

DATA = json.load(open("cot_tasks.json", encoding="utf-8")) if os.path.exists("cot_tasks.json") else _EMBEDDED
GOLD = DATA["gold"]
TASK_IDS = [t["id"] for t in DATA["mwp"]] + [t["id"] for t in DATA["pol"]]
print(f"Loaded {len(TASK_IDS)} tasks and {len(GOLD)} gold answers.")


In [ ]:
# GIVEN - the offline teaching provider. Do not edit this cell.
import hashlib
import re

# Per-task ground truth, the plausible wrong answer a shallow reader lands on,
# and a short worked rationale for the reasoning path.
_TASKS = {
    "M01": {"gold": 20, "shortcut": 20,
            "steps": "18+12=30 marbles; remove 7+3=10; 30-10=20."},
    "M02": {"gold": 54, "shortcut": 55,
            "steps": "Distance 120+150=270 km; time 2+3=5 h; 270 divided by 5 is 54 km per hour."},
    "M03": {"gold": 38, "shortcut": 38,
            "steps": "5*8=40 pens; 40-9=31; 31+7=38."},
    "M04": {"gold": 4, "shortcut": 12,
            "steps": "Three quarters of 48 is 36 sold; plus 8 is 44; 48-44=4 left."},
    "M05": {"gold": 19, "shortcut": 15,
            "steps": "x+y=30 and x=y+8; 2y+8=30; y=11; larger x=19."},
    "P01": {"gold": "approve", "shortcut": "approve",
            "steps": "21:40 is within 06:00-22:00 and 47<=50; both conditions met."},
    "P02": {"gold": "reimburse_meal_only", "shortcut": "reimburse_meal_only",
            "steps": "Meal 14 qualifies; alcohol 6 never reimbursable; meal only."},
    "P03": {"gold": "deny", "shortcut": "deny",
            "steps": "No IT pre-approval before purchase; policy fails; deny."},
    "P04": {"gold": "deny", "shortcut": "approve",
            "steps": "Approval was the day after purchase, not pre-approval; deny."},
    "P05": {"gold": "deny", "shortcut": "approve",
            "steps": "Fare 32 plus tip 9 is 41, which exceeds 40; deny."},
}

# Items a shallow no-CoT pass gets wrong (multi-step or trap items).
_HARD_NO_COT = {"M02", "M04", "M05", "P04", "P05"}
# Items the CoT path can still slip on when temperature > 0.
_SLIP_PRONE = {"M02", "M04", "P05"}
_SLIP_PROB = 0.34

_TRIGGER_RE = re.compile(r"step[\s\-]*by[\s\-]*step", re.IGNORECASE)
_HIDE_RE = re.compile(
    r"do not include|don't include|silently|privately|no rationale|"
    r"without.*reason|only.*final|hide.*reason|exclude.*reason",
    re.IGNORECASE,
)


def _seed_int(*parts) -> int:
    """Stable, process-independent seed from strings (Python hash() is salted)."""
    key = "|".join(str(p) for p in parts).encode("utf-8")
    return int.from_bytes(hashlib.sha256(key).digest()[:8], "big")


def _uniform(*parts) -> float:
    """Deterministic pseudo-uniform in [0, 1) from a stable seed."""
    return (_seed_int(*parts) % 1_000_000) / 1_000_000.0


class TeachingProvider:
    def __init__(self, tasks_by_id=None):
        self.tasks = tasks_by_id or _TASKS

    def complete(self, prompt: str, task_ids, temperature: float = 0.0,
                 run: int = 0) -> dict:
        """Return a STRICT-JSON-style dict for the given task ids.

        Reasoning is enabled when the prompt contains a step-by-step trigger.
        A rationale is returned when the prompt asks for one and does not also
        instruct the model to hide its reasoning.
        """
        cot_on = bool(_TRIGGER_RE.search(prompt))
        hide = bool(_HIDE_RE.search(prompt))
        want_rationale = ("rationale" in prompt.lower()) and not hide

        records = []
        for tid in task_ids:
            spec = self.tasks[tid]
            if not cot_on:
                answer = spec["shortcut"] if tid in _HARD_NO_COT else spec["gold"]
            else:
                slipped = (
                    temperature > 0.0
                    and tid in _SLIP_PRONE
                    and _uniform(tid, run, round(temperature, 3)) < _SLIP_PROB
                )
                answer = spec["shortcut"] if slipped else spec["gold"]
            rec = {"id": tid, "final_answer": answer}
            if want_rationale:
                rec["rationale"] = spec["steps"]
            records.append(rec)

        stats = {"count": len(records)}
        if run:
            stats["sampling_run"] = run
        return {"records": records, "stats": stats}


In [ ]:
# GIVEN - soft check helpers. Do not edit this cell.
# These NEVER raise: a missing or wrong implementation shows as [FAIL], not a crash,
# so you can run the whole notebook end to end at any point.
_CHECKS = []

def check(label, passed, detail=""):
    passed = bool(passed)
    _CHECKS.append((label, passed))
    tag = "PASS" if passed else "FAIL"
    line = f"[{tag}] {label}"
    if detail and not passed:
        line += f"   ->  {detail}"
    print(line)
    return passed

def expect(label, fn, expected):
    """Call fn() and check equality with expected, catching any error."""
    try:
        got = fn()
    except NotImplementedError:
        return check(label, False, "not implemented yet")
    except Exception as e:
        return check(label, False, f"raised {type(e).__name__}: {e}")
    return check(label, got == expected, f"got {got!r}, expected {expected!r}")

def expect_true(label, fn):
    try:
        got = bool(fn())
    except NotImplementedError:
        return check(label, False, "not implemented yet")
    except Exception as e:
        return check(label, False, f"raised {type(e).__name__}: {e}")
    return check(label, got, "predicate returned False")

def check_summary():
    passed = sum(1 for _, ok in _CHECKS if ok)
    print(f"\n===== {passed} / {len(_CHECKS)} checks passing =====")


In [ ]:
# GIVEN - provider handle, the two baseline prompts, and a sampling helper.
# You write the PRIVATE prompt yourself in Part D. Do not edit this cell.
prov = TeachingProvider()

def sample(prompt, temperature=0.0, run=0):
    """Send a prompt to the provider for all TASK_IDS and return its JSON dict.
    temperature > 0 makes the reasoning path occasionally slip (needed for
    self-consistency); run is a stable sample index for reproducibility."""
    return prov.complete(prompt, TASK_IDS, temperature=temperature, run=run)

PROMPT_NO_COT = """<INSTRUCTION>
Solve each task and return STRICT JSON per OUTPUT_CONTRACT. Return only the JSON.
</INSTRUCTION>
<OUTPUT_CONTRACT>
{"records": [{"id": "<ID>", "final_answer": "<value_or_label>"}], "stats": {"count": "<int>"}}
</OUTPUT_CONTRACT>
"""

PROMPT_COT_EXPOSED = """<INSTRUCTION>
Let's think step by step. For each task compute intermediate quantities, then
give the final answer. Return STRICT JSON per OUTPUT_CONTRACT. Keep each
rationale to two to five short steps. Return only the JSON.
</INSTRUCTION>
<OUTPUT_CONTRACT>
{"records": [{"id": "<ID>", "final_answer": "<value_or_label>", "rationale": "<short steps>"}], "stats": {"count": "<int>"}}
</OUTPUT_CONTRACT>
"""
print("Provider ready. Baseline prompts loaded.")


## Part A - Warm up and inspect the data

Two mini task families: Math Word Problems (`M01..M05`) and Policy Logic (`P01..P05`). Every item has a single gold answer. Numeric answers are integers; policy answers are one of the labels `approve`, `deny`, or `reimburse_meal_only`.

Run the cell below and read a few tasks so you know what the provider is answering.

In [ ]:
# GIVEN - inspect the dataset.
for t in (DATA["mwp"] + DATA["pol"])[:4]:
    print(t["id"], "->", t["text"])
print("...")
print("gold:", GOLD)

## Part B - Baselines: no-CoT versus think step by step

You will build two small pieces of the evaluation harness, then use them to compare a plain prompt against a CoT prompt.

The two prompts are already loaded: `PROMPT_NO_COT` and `PROMPT_COT_EXPOSED`. First, the answer comparison.

### TODO 1 - `normalize_answer`

In [ ]:
def normalize_answer(ans):
    """Canonicalise one answer so predictions and gold compare cleanly.

    Contract:
      - bool stays bool; int/float stay numeric and unchanged.
      - a numeric string ("54", "-3", "4.0") becomes the matching int or float.
      - any other string is trimmed of surrounding whitespace and lowercased.
      - anything else is returned unchanged.
    """
    # TODO: implement per the contract above.
    raise NotImplementedError

In [ ]:
# CHECK - normalize_answer
expect("int passes through", lambda: normalize_answer(54), 54)
expect("numeric string to int", lambda: normalize_answer("54"), 54)
expect("decimal string to float", lambda: normalize_answer("4.0"), 4.0)
expect("negative numeric string", lambda: normalize_answer("-3"), -3)
expect("label trimmed and lowered", lambda: normalize_answer("  Approve "), "approve")
expect("missing answer stays None", lambda: normalize_answer(None), None)

### TODO 2 - `score`

In [ ]:
def score(pred, gold):
    """Score a prediction dict against the gold map.

    Returns (correct, total, per_item) where per_item is a list of tuples
    (id, normalized_pred, normalized_gold, is_correct), one per gold key.
    Use normalize_answer on both sides before comparing. A gold id with no
    matching record counts as incorrect.
    """
    # TODO: implement per the contract above.
    raise NotImplementedError

In [ ]:
# CHECK - score against the two baselines
no_cot = sample(PROMPT_NO_COT, temperature=0.0)
cot_single = sample(PROMPT_COT_EXPOSED, temperature=0.55, run=1)

def acc(pred):
    c, t, _ = score(pred, GOLD)
    return c

expect("no-CoT scores 5 of 10", lambda: acc(no_cot), 5)
expect("single CoT scores 9 of 10", lambda: acc(cot_single), 9)

# Show the per-item breakdown for the single CoT run.
try:
    _, _, per = score(cot_single, GOLD)
    print()
    for k, y, g, ok in per:
        print(f"  {k}: pred={y!r:>22} gold={g!r:>22} {'ok' if ok else 'MISS'}")
except Exception as e:
    print("implement score to see the breakdown:", e)

## Part C - Self consistency: sample and select

A single CoT run can slip on the hardest items. Self consistency draws several samples at a non-zero temperature and takes a majority vote, which cancels out independent slips.

Build the consensus function, then run it over five samples.

### TODO 3 - `self_consistency`

In [ ]:
def self_consistency(runs):
    """Build a majority-vote consensus over a list of run dicts.

    For each task id, tally votes across runs (compare answers as strings).
    The winner is the answer with the most votes. Break a tie deterministically
    by choosing the tied answer that was seen first across the runs (earliest
    stable winner). Each output record carries an "agreement" field equal to
    winning_votes / total_votes, rounded to 3 places.

    Return {"records": [...], "stats": {"count": <int>, "runs": <int>}}.
    """
    # TODO: implement per the contract above.
    raise NotImplementedError

In [ ]:
# CHECK - self consistency over five samples
runs = [sample(PROMPT_COT_EXPOSED, temperature=0.55, run=k) for k in range(1, 6)]
try:
    consensus = self_consistency(runs)          # raises until TODO 3 is done
except NotImplementedError:
    consensus = {"records": [], "stats": {"count": 0, "runs": 0}}
consensus_scored = {"records": [{"id": r["id"], "final_answer": r["final_answer"]} for r in consensus["records"]],
                    "stats": {"count": len(consensus["records"])}}

expect("consensus scores 10 of 10", lambda: acc(consensus_scored), 10)
expect_true("every record carries an agreement value",
            lambda: bool(consensus["records"]) and all("agreement" in r for r in consensus["records"]))

# Per-item agreement: the items that ever slipped show agreement below 1.0.
try:
    print()
    for r in consensus["records"]:
        print(f"  {r['id']}: {r['final_answer']!r:>22}  agreement={r['agreement']}")
except Exception as e:
    print("implement self_consistency to see agreement:", e)

## Part D - Private versus exposed rationales

Exposed rationales are useful for audits and teaching. In production you often want the reasoning kept out of the response, both to avoid leaking chain of thought and to save tokens. You will build a guard that fails if a private-mode output leaks a rationale, then write the private prompt yourself.

The guard runs first.

### TODO 4 - `guard_private`

In [ ]:
def guard_private(pred):
    """Return the list of record ids that leaked a rationale in a private-mode
    output. An empty list means the guard passed (no reasoning was exposed)."""
    # TODO: implement per the contract above.
    raise NotImplementedError

### TODO 6 - write the private CoT prompt

This is the prompt engineering task. Build a prompt that keeps reasoning on but returns only the final answer.

In [ ]:
# TODO 6: write a CoT prompt that reasons step by step but returns ONLY the
# final answer, with no rationale in the output. Starting from PROMPT_COT_EXPOSED,
# keep a step-by-step trigger (so reasoning stays on) but instruct the model to
# think privately and drop the rationale from the OUTPUT_CONTRACT.
# Your prompt passes when: the private-mode guard finds no leaked rationale AND
# the private run still scores 10 / 10 (reasoning stayed on).
PROMPT_COT_PRIVATE = """"""  # <-- replace with your prompt

In [ ]:
# CHECK - private versus exposed
exposed = sample(PROMPT_COT_EXPOSED, temperature=0.0)
private = sample(PROMPT_COT_PRIVATE, temperature=0.0)

expect("guard passes on private output (no leak)", lambda: guard_private(private), [])
expect_true("guard flags the exposed output (rationale present)",
            lambda: len(guard_private(exposed)) > 0)
expect("private output still scores 10 of 10 (reasoning stayed on)",
       lambda: acc(private), 10)

## Part E - Acceptance and reflection

One more harness piece: a structural gate that checks any artifact obeys the output contract before you trust or submit it.

### TODO 5 - `acceptance`

In [ ]:
def acceptance(pred):
    """Validate the output contract. Return (ok: bool, problems: list[str]).

    Fail (and report a message) if any of these hold:
      - pred is not a dict, or is missing "records" or "stats".
      - records is not a non-empty list.
      - stats["count"] does not equal the number of records.
      - any record is missing "id" or "final_answer".
    ok is True only when problems is empty.
    """
    # TODO: implement per the contract above.
    raise NotImplementedError

In [ ]:
# CHECK - acceptance
expect("no-CoT artifact passes acceptance", lambda: acceptance(no_cot)[0], True)
expect("consensus artifact passes acceptance", lambda: acceptance(consensus_scored)[0], True)

broken = {"records": [{"id": "M01"}], "stats": {"count": 5}}  # missing final_answer, wrong count
expect("a malformed artifact fails acceptance", lambda: acceptance(broken)[0], False)
expect_true("acceptance explains why it failed", lambda: len(acceptance(broken)[1]) > 0)

In [ ]:
# GIVEN - final tally
check_summary()

## Reflect

Answer briefly, in a markdown cell or your notes.

1. **Accuracy.** No-CoT scored 5 of 10 here. Look at which five items it missed. What do they have in common that a single-step answer gets wrong?
2. **Self consistency.** Consensus reached 10 of 10 while single runs did not. Which items had agreement below 1.0, and why does majority voting help exactly those items?
3. **Cost.** Self consistency multiplied the number of model calls by five. When is that worth it, and when would you keep a single call?
4. **Rationale visibility.** Exposed rationales help you debug and audit. Name one concrete risk of returning them in a production response, and one thing the private guard does not protect you from.

### Self assessment (0 to 2 each, target 10 of 14)
- CoT trigger used correctly.
- Self consistency runs at least five samples and builds a consensus.
- Private and exposed prompts behave correctly; private does not leak.
- Outputs pass acceptance.
- Accuracy measured across no-CoT, single CoT, and consensus.
- Reflection covers accuracy, latency, tokens, and interpretability.
- Code is readable and the checks pass.

## Stretch goals (optional)

1. **Calibration curve.** Compute consensus accuracy for K equals 1 through 5 and describe the shape. Watch what happens at even K.
2. **Answer then verify.** Take one CoT sample, run a second deterministic pass as a verifier, and where they disagree keep the verifier. Measure accuracy before and after.
3. **Adversarial phrasing.** Reword a couple of task prompts to be misleading and see whether CoT and consensus still hold up.

## Key takeaways

- CoT structures the intermediate steps, which is what fixes multi-step and trap items.
- Self consistency samples and votes, cancelling independent slips; odd K avoids ties.
- Keep rationales private in production and expose them for audits and learning, and guard the boundary.
- Reasoning costs tokens and latency, and a rationale can still be wrong, so verify.